# Imports

In [1]:
import pandas as pd
import numpy as np

# Classification method 1
import re
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt

# Model imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


In [2]:
evaluate_df = pd.read_csv('data/gold_set.csv')
evaluate_df.head()

,description,category_code,category_name
0,"Доставка груза 5 тонн, маршрут Шымкент-Тараз",49.41,Услуги перевозки грузов автотранспортом
1,Дизельное топливо 5000 л,19.20,Топливо и ГСМ
2,"Офисная мебель: столы, стулья — 15 рабочих мест",31.01,Мебель и офисное оборудование
3,Услуги: лекарственные препараты по заявке №21,21.20,Медицинские товары и препараты
4,хлебобулочные изделия — партия №14,10.71,Продукты питания (опт)


In [3]:
test_df = pd.read_csv('data/transactions_cleaned.csv')

test_df.head()

,sender_id,receiver_id,date,amount_kzt,description,doc_type,sender_valid,receiver_valid,both_valid_id,year_month,description_clean
0,341113577953,421128527724,2024-10-05,18579.93,"Поставка канцелярских товаров (бумага А4, ручк...",INVOICE,1,1,1,2024-10,"поставка канцелярских товаров (бумага а4, ручк..."
1,950908633188,821120551797,2025-07-29,1706226.33,Поставка медикаментов по списку,INVOICE,1,1,1,2025-07,поставка медикаментов по списку
2,500817613929,830425668259,2025-05-17,286266.87,"СИЗ: перчатки, очки, респираторы",INVOICE,1,1,1,2025-05,"сиз: перчатки, очки, респираторы"
3,110824699080,220312376061,2024-02-08,996569.04,Грузоперевозка щебня по г. Алматы,INVOICE,1,1,1,2024-02,грузоперевозка щебня по г. алматы
4,910816595223,110910187697,2024-10-20,120351.41,ГСМ за февраль 2025,INVOICE,1,1,1,2024-10,гсм за февраль 2025


In [4]:
train_df = pd.read_json('data/categories.json', convert_axes=False).T
train_df.reset_index(inplace=True)
train_df.columns = ['code', 'name', 'keywords']
train_df

,code,name,keywords
0,49.41,Услуги перевозки грузов автотранспортом,"[перевозк, доставка груз, транспорт, грузопере..."
1,17.23,Бумажные канцелярские принадлежности,"[канцеляр, бумага, ручк, тетрад, папк, степлер]"
2,62.01,Разработка ПО и IT-услуги,"[разработк, программн, сайт, web, софт, IT, ИТ..."
3,68.20,Аренда офисных помещений,"[аренд, офис, помещен, жалд]"
4,35.30,"Коммунальные услуги (электро, газ, тепло)","[коммунальн, электроэнерг, отоплен, газ, ХВС, ..."
5,69.10,Юридические услуги,"[юридическ, юрист, адвокат, правов, консультац]"
6,69.20,Бухгалтерские и аудиторские услуги,"[бухгалтер, аудит, налогов, отчётност, отчетност]"
7,23.61,Строительные материалы,"[цемент, кирпич, арматур, щебен, стройматериал..."
8,10.71,Продукты питания (опт),"[продукт, хлеб, молок, мяс, крупа, сахар, мука..."
9,19.20,Топливо и ГСМ,"[дизель, бензин, ГСМ, топлив, АИ-92, АИ-95, ДТ]"


In [5]:
def precompile_keyword(categories: pd.DataFrame) -> list:
    """Простое вхождение подстроки — без границ слова"""
    compiled = []
    for _, row in categories.iterrows():
        keywords = row['keywords']
        if not keywords:
            continue
        pattern = re.compile('|'.join(re.escape(kw.lower()) for kw in keywords))
        compiled.append({
            'code': row['code'],
            'name': row['name'],
            'pattern': pattern,
            'kw_count': len(keywords)
        })
    return compiled


def precompile_regex(categories: pd.DataFrame) -> list:
    compiled = []
    for _, row in categories.iterrows():
        keywords = row['keywords']
        if not keywords:
            continue
        
        parts = []
        for kw in keywords:
            esc = re.escape(kw.lower())
            parts.append(r'(?<![а-яёА-ЯЁa-zA-Z])' + esc + r'(?![а-яёА-ЯЁa-zA-Z])')
        
        pattern = re.compile('|'.join(parts))
        compiled.append({
            'code': row['code'],
            'name': row['name'],
            'pattern': pattern,
            'kw_count': len(keywords)
        })
    return compiled

compiled_keyword = precompile_keyword(train_df)
compiled_regex = precompile_regex(train_df)

In [6]:
def classify_keyword_fast(text: str, compiled: list) -> tuple:
    if pd.isna(text):
        return None, "undefined", 0.0
    
    text_lower = text.lower()
    scores = []

    for cat in compiled:
        matches = len(cat['pattern'].findall(text_lower))
        scores.append((matches / cat['kw_count'], cat['code'], cat['name']))

    scores.sort(reverse=True)
    if scores[0][0] == 0:
        return None, "undefined", 0.0

    best, second = scores[0], scores[1]
    confidence = best[0] / (best[0] + second[0] + 1e-9)
    return best[1], best[2], round(confidence, 3)

In [ ]:
test_classify_keyword = test_df.copy()
test_classify_keyword[['pred_code', 'pred_name', 'confidence']] = test_df['description'].apply(
    lambda x: pd.Series(classify_keyword_fast(x, compiled_keyword))
)

test_classify_regex = test_df.copy()
test_classify_regex[['pred_code', 'pred_name', 'confidence']] = test_df['description'].apply(
    lambda x: pd.Series(classify_keyword_fast(x, compiled_regex))
)

In [ ]:
test_classify_regex.head()

In [ ]:
test_classify_keyword.head()

In [ ]:
print(evaluate_df.columns.tolist())
evaluate_df.head()

In [ ]:
THRESHOLD = 0.6

evaluate_df['pred_keyword'] = evaluate_df['description'].apply(
    lambda x: classify_keyword_fast(x, compiled_keyword)[1]
)
evaluate_df['conf_keyword'] = evaluate_df['description'].apply(
    lambda x: classify_keyword_fast(x, compiled_keyword)[2]
)

evaluate_df['pred_regex'] = evaluate_df['description'].apply(
    lambda x: classify_keyword_fast(x, compiled_regex)[1]
)
evaluate_df['conf_regex'] = evaluate_df['description'].apply(
    lambda x: classify_keyword_fast(x, compiled_regex)[2]
)

y_true = evaluate_df['category_name']

print("=" * 60)
print("📊 KEYWORD baseline")
print("=" * 60)
print(classification_report(y_true, evaluate_df['pred_keyword'], zero_division=0))

print("=" * 60)
print("📊 REGEX baseline")
print("=" * 60)
print(classification_report(y_true, evaluate_df['pred_regex'], zero_division=0))


In [ ]:
labels = sorted(y_true.unique())

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

for ax, preds, title in zip(
    axes,
    [evaluate_df['pred_keyword'], evaluate_df['pred_regex']],
    ['Keyword Baseline', 'Regex Baseline']
):
    cm = confusion_matrix(y_true, preds, labels=labels)
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(
        ax=ax, xticks_rotation=45, colorbar=False, cmap='Blues'
    )
    ax.set_title(title, fontsize=13)

plt.tight_layout()
plt.savefig('confusion_matrix_comparison.png', dpi=150)
plt.show()

In [ ]:
for method, pred_col, conf_col in [
    ('Keyword', 'pred_keyword', 'conf_keyword'),
    ('Regex',   'pred_regex',   'conf_regex'),
]:
    df = evaluate_df.copy()
    df['correct'] = df[pred_col] == y_true
    df['needs_review'] = df[conf_col] < THRESHOLD

    auto    = df[~df['needs_review']]
    review  = df[df['needs_review']]

    print(f"\n{'─'*50}")
    print(f"🔎 {method} | threshold = {THRESHOLD}")
    print(f"  Reliable (≥{THRESHOLD}):  {len(auto)}  — accuracy: {auto['correct'].mean():.3f}")
    print(f"  To check (<{THRESHOLD}): {len(review)}  — accuracy: {review['correct'].mean():.3f}")


In [ ]:
summary = pd.DataFrame({
    'Method':    ['Keyword', 'Regex'],
    'Accuracy':  [
        (evaluate_df['pred_keyword'] == y_true).mean(),
        (evaluate_df['pred_regex']   == y_true).mean(),
    ],
    'F1 (macro)': [
        f1_score(y_true, evaluate_df['pred_keyword'], average='macro', zero_division=0),
        f1_score(y_true, evaluate_df['pred_regex'],   average='macro', zero_division=0),
    ],
    'Precision (macro)': [
        precision_score(y_true, evaluate_df['pred_keyword'], average='macro', zero_division=0),
        precision_score(y_true, evaluate_df['pred_regex'],   average='macro', zero_division=0),
    ],
    'Recall (macro)': [
        recall_score(y_true, evaluate_df['pred_keyword'], average='macro', zero_division=0),
        recall_score(y_true, evaluate_df['pred_regex'],   average='macro', zero_division=0),
    ],
    'На проверку': [
        (evaluate_df['conf_keyword'] < THRESHOLD).sum(),
        (evaluate_df['conf_regex']   < THRESHOLD).sum(),
    ],
})

print("\n", summary.to_string(index=False))

---

# TF-IDF + LogReg

In [ ]:
print(test_df.columns.tolist())
print(test_df.shape)
test_df.head()

In [ ]:
test_df['pseudo_label'] = test_df['description'].apply(
    lambda x: classify_keyword_fast(x, compiled_keyword)[1]
)

test_df['confidence'] = test_df['description'].apply(
    lambda x: classify_keyword_fast(x, compiled_keyword)[2]
)

train_ml = test_df[
    (test_df['pseudo_label'] != 'undefined') &
    (test_df['confidence'] >= 0.6)
].copy()

print(f"Learning examples: {len(train_ml)}")
print(train_ml['pseudo_label'].value_counts())

In [ ]:
X_train = train_ml['description_clean']
y_train = train_ml['pseudo_label']

X_test  = evaluate_df['description']
y_test  = evaluate_df['category_name']

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),    # unigrams + bigrams
        min_df=3,              # ignore rare tokens
        max_features=50000,    # top 50k tokens
        sublinear_tf=True,     # log(tf) smoothing
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=5,
        class_weight='balanced',  # handle class imbalance
        random_state=42,
    ))
])

pipeline.fit(X_train, y_train)
print("Model trained!")


y_pred_ml = pipeline.predict(X_test)
y_proba_ml = pipeline.predict_proba(X_test)
confidence_ml = y_proba_ml.max(axis=1)

evaluate_df['pred_ml'] = y_pred_ml
evaluate_df['conf_ml'] = confidence_ml.round(3)
evaluate_df['needs_review_ml'] = evaluate_df['conf_ml'] < THRESHOLD


print("=" * 60)
print("TF-IDF + LogReg")
print("=" * 60)
print(classification_report(y_test, y_pred_ml, zero_division=0))

summary = pd.DataFrame({
    'Method': ['Keyword', 'Regex', 'TF-IDF + LogReg'],
    'Accuracy': [
        (evaluate_df['pred_keyword'] == y_test).mean(),
        (evaluate_df['pred_regex']   == y_test).mean(),
        (evaluate_df['pred_ml']      == y_test).mean(),
    ],
    'F1 (macro)': [
        f1_score(y_test, evaluate_df['pred_keyword'], average='macro', zero_division=0),
        f1_score(y_test, evaluate_df['pred_regex'],   average='macro', zero_division=0),
        f1_score(y_test, evaluate_df['pred_ml'],      average='macro', zero_division=0),
    ],
    'Precision (macro)': [
        precision_score(y_test, evaluate_df['pred_keyword'], average='macro', zero_division=0),
        precision_score(y_test, evaluate_df['pred_regex'],   average='macro', zero_division=0),
        precision_score(y_test, evaluate_df['pred_ml'],      average='macro', zero_division=0),
    ],
    'Recall (macro)': [
        recall_score(y_test, evaluate_df['pred_keyword'], average='macro', zero_division=0),
        recall_score(y_test, evaluate_df['pred_regex'],   average='macro', zero_division=0),
        recall_score(y_test, evaluate_df['pred_ml'],      average='macro', zero_division=0),
    ],
    'Needs review': [
        (evaluate_df['conf_keyword'] < THRESHOLD).sum(),
        (evaluate_df['conf_regex']   < THRESHOLD).sum(),
        (evaluate_df['needs_review_ml']).sum(),
    ],
})

print("\n", summary.to_string(index=False))

auto = evaluate_df[~evaluate_df['needs_review_ml']]
review = evaluate_df[evaluate_df['needs_review_ml']]

print(f"\n{'─'*50}")
print(f"TF-IDF + LogReg | threshold = {THRESHOLD}")
print(f"Confident (≥{THRESHOLD}): {len(auto)}  — accuracy: {(auto['pred_ml'] == auto['category_name']).mean():.3f}")
print(f"Needs review (<{THRESHOLD}): {len(review)}  — accuracy: {(review['pred_ml'] == review['category_name']).mean():.3f}")

---

# Указать 2 пары категорий с систематической путаницей

Systematic Confusion Analysis - Keyword & TF-IDF + LogReg Classifiers

Two pairs of categories showed systematic misclassification across both models. The first pair, Consulting Services vs. Legal Services, produced the lowest F1 score of 0.40 for the consulting class. Both categories share overlapping vocabulary such as "консультац", "сопровожден", and "правов", making it difficult for both keyword matching and TF-IDF to distinguish them. The recommended fix is to add discriminative priority rules — if strong legal-specific terms like "иск", "суд", or "арбитраж" are present, override the prediction toward Legal Services regardless of model output.

The second pair, Accounting & Audit Services vs. Banking & Financial Services, achieved only F1 0.50 for the accounting class. Terms like "финансов", "отчётност", and "налог" frequently appear in both contexts, causing the model to conflate the two. The most reliable fix here is targeted manual labeling — the current gold set contains only 10 examples per class, which is insufficient for semantically similar categories. Adding 50–100 hand-labeled examples focused specifically on these two confused classes would likely push F1 above 0.95.

In both cases, the root cause is the same: semantically adjacent categories with insufficient discriminative signal in the training data. A combination of hard priority rules for the consulting/legal pair and expanded labeled data for the accounting/banking pair is the recommended remediation path.